# 28 — Caching RAG

> **Tier 7 | Production**

## What is Caching RAG?

Every RAG query makes at least two expensive calls: one to the embedding API and one to
the LLM. In production many queries are repeated or near-duplicates. **Caching RAG**
eliminates redundant API calls with a two-level cache:

### Cache levels

| Level | Key | Hit condition | Saves |
|-------|-----|--------------|-------|
| **L1 — Exact cache** | SHA-256 of query string | Identical query | Embed + retrieve + generate |
| **L2 — Semantic cache** | Cosine similarity of query embedding | Similarity ≥ threshold | Retrieve + generate |

L1 is checked first (free — just a dict lookup). L2 requires embedding the query but
skips retrieval and generation if a semantically similar answer already exists.

### Cache metrics tracked

| Metric | Definition |
|--------|------------|
| `exact_hits` | Queries served from L1 |
| `semantic_hits` | Queries served from L2 |
| `misses` | Full pipeline run |
| `embed_calls_saved` | Embedding API calls avoided |
| `llm_calls_saved` | LLM API calls avoided |

## PDF Framework: pymupdf `get_text("rawdict")`

This notebook uses **pymupdf** with `get_text("rawdict")` — the raw dictionary mode
that returns character-level bounding boxes. Unlike `"dict"` (NB 19) which groups into
spans/lines, `"rawdict"` exposes individual character glyphs. We use it here to
demonstrate the mode and extract text by concatenating character strings.

| pymupdf mode | Granularity | Used in |
|-------------|------------|--------|
| `get_text("text")` | Page string | NB 25 |
| `get_text("blocks")` | Block tuples | NB 22 |
| `get_text("dict")` | Span/line/font | NB 19 |
| **`get_text("rawdict")`** | **Char-level glyphs** | **NB 28** |

## Flow Diagram


In [ ]:
from IPython.display import display as _d, HTML as _H
_d(_H("""
<script src="https://cdn.jsdelivr.net/npm/mermaid@10/dist/mermaid.min.js"></script>
<div class="mermaid" style="background:#fff;padding:16px;border-radius:8px;">
flowchart TD
    subgraph INDEX ["🔵  INDEXING — pymupdf rawdict"]
        PDF(["📄 climate.pdf"])
        PDF --> MU["fitz get_text('rawdict')\nchar-level extraction"]
        MU --> CH["Text chunks"]
        CH --> EMB["Embed — Titan V2"]
        EMB --> QDRANT[("Qdrant\ncaching_rag_28")]
    end
    subgraph CACHE ["⚡  TWO-LEVEL CACHE"]
        Q(["❓ Query"])
        Q --> L1{"L1: Exact\nhash match?"}
        L1 -->|HIT| ANS1(["✅ Cached answer"])
        L1 -->|MISS| EMBQ["Embed query"]
        EMBQ --> L2{"L2: Semantic\nsim ≥ threshold?"}
        L2 -->|HIT| ANS2(["✅ Semantic answer"])
        L2 -->|MISS| RET["Retrieve from Qdrant"]
        RET --> GEN["LLM Generation"]
        GEN --> STORE["Store in L1 + L2"]
        STORE --> ANS3(["✅ Fresh answer"])
    end
    style INDEX fill:#dbeafe,stroke:#3b82f6,color:#1e3a5f
    style CACHE fill:#fdf4ff,stroke:#9333ea,color:#3b0764
</div>
<script>mermaid.initialize({startOnLoad:true,theme:'default',flowchart:{curve:'basis'}});</script>
"""))


## Step 1 — Install & Import Dependencies

In [ ]:
import subprocess, sys
packages = ["boto3", "qdrant-client", "opensearch-py", "requests-aws4auth",
            "pymupdf", "python-dotenv"]
subprocess.check_call([sys.executable, "-m", "pip", "install", "--quiet"] + packages)
print("All packages ready.")


In [ ]:
import os, json, time, uuid, hashlib
from typing import List, Dict, Optional
from dotenv import load_dotenv

import boto3
import fitz   # pymupdf
from qdrant_client import QdrantClient
from qdrant_client.models import Distance, VectorParams, PointStruct

load_dotenv(r"C:\Users\Administrator\RAG\.env", override=True)
print("Imports OK")


## Step 2 — Configuration

In [ ]:
AWS_REGION      = os.getenv("AWS_DEFAULT_REGION", "us-east-1")
EMBEDDING_MODEL = "amazon.titan-embed-text-v2:0"
LLM_MODEL       = "us.anthropic.claude-sonnet-4-6"

QDRANT_URL     = os.getenv("QDRANT_URL", "")
QDRANT_API_KEY = os.getenv("QDRANT_API_KEY", "")

COLLECTION_NAME    = "caching_rag_28"
EMBEDDING_DIM      = 1024
CHUNK_SIZE         = 500
CHUNK_OVERLAP      = 50
TOP_K              = 5
SEMANTIC_THRESHOLD = 0.75   # cosine similarity threshold for L2 cache hit

PDF_PATH = r"C:\Users\Administrator\RAG\data\climate.pdf"

print(f"Collection         : {COLLECTION_NAME}")
print(f"PDF                : {PDF_PATH}  (exists={os.path.exists(PDF_PATH)})")
print(f"Semantic threshold : {SEMANTIC_THRESHOLD}")


## Step 3 — Qdrant Setup

In [ ]:
def make_qdrant(url='', api_key=''):
    if url:
        try:
            kw = {'url': url}
            if api_key: kw['api_key'] = api_key
            client = QdrantClient(**kw)
            client.get_collections()
            print(f'Qdrant Cloud: {url}')
            return client
        except Exception as e:
            print(f'Qdrant Cloud unavailable ({e}), falling back...')
    print('Using Qdrant in-memory')
    return QdrantClient(':memory:')


qdrant = make_qdrant(QDRANT_URL, QDRANT_API_KEY)

if COLLECTION_NAME in [c.name for c in qdrant.get_collections().collections]:
    qdrant.delete_collection(COLLECTION_NAME)
qdrant.create_collection(
    COLLECTION_NAME,
    vectors_config=VectorParams(size=EMBEDDING_DIM, distance=Distance.COSINE))
print(f'Created "{COLLECTION_NAME}" (dim={EMBEDDING_DIM})')


## Step 4 — Bedrock Helpers

In [ ]:
bedrock_rt = boto3.client('bedrock-runtime', region_name=AWS_REGION)

def embed_text(text: str) -> List[float]:
    body = json.dumps({"inputText": text, "dimensions": EMBEDDING_DIM, "normalize": True})
    resp = bedrock_rt.invoke_model(
        modelId=EMBEDDING_MODEL, body=body,
        contentType="application/json", accept="application/json")
    return json.loads(resp["body"].read())["embedding"]

def embed_batch(texts: List[str], label: str = '') -> List[List[float]]:
    out = []
    for i, t in enumerate(texts):
        out.append(embed_text(t))
        if (i + 1) % 20 == 0:
            print(f'  {label} {i+1}/{len(texts)}')
        time.sleep(0.04)
    return out

def call_llm(system: str, user_content: str, max_tokens: int = 1024) -> str:
    body = json.dumps({
        "anthropic_version": "bedrock-2023-05-31",
        "max_tokens": max_tokens,
        "system": system,
        "messages": [{"role": "user", "content": user_content}],
    })
    resp = bedrock_rt.invoke_model(
        modelId=LLM_MODEL, body=body,
        contentType="application/json", accept="application/json")
    return json.loads(resp["body"].read())["content"][0]["text"]

test_emb = embed_text("caching rag pymupdf rawdict test")
print(f"Embedding OK — dim={len(test_emb)}")


## Step 5 — PDF Loading with pymupdf `get_text("rawdict")`

`get_text("rawdict")` returns the page as a nested dict with character-level `origin`
coordinates and glyph info. The structure is:
`page → blocks → lines → spans → chars`.
Each `char` dict has a `c` key (the character string). We concatenate characters
within each span to reconstruct words, then join spans and lines into page text.


In [ ]:
def recursive_split(text: str, size: int, overlap: int) -> List[str]:
    if len(text) <= size:
        return [text] if text.strip() else []
    chunks, start = [], 0
    while start < len(text):
        chunk = text[start:start + size].strip()
        if chunk:
            chunks.append(chunk)
        start += size - overlap
    return chunks


def extract_page_text_rawdict(page) -> str:
    raw   = page.get_text("rawdict")
    parts = []
    for block in raw.get("blocks", []):
        if block.get("type") != 0:   # skip image blocks
            continue
        for line in block.get("lines", []):
            for span in line.get("spans", []):
                # each char dict has key 'c'
                span_text = "".join(ch.get("c", "") for ch in span.get("chars", []))
                if span_text.strip():
                    parts.append(span_text)
    return " ".join(parts)


def load_pdf_pymupdf_rawdict(path: str):
    doc     = fitz.open(path)
    n_pages = len(doc)
    chunks  = []

    for page_num in range(n_pages):
        text = ' '.join(extract_page_text_rawdict(doc[page_num]).split()).strip()
        if not text:
            continue
        for sub in recursive_split(text, CHUNK_SIZE, CHUNK_OVERLAP):
            chunks.append({
                'text'      : sub,
                'page'      : page_num + 1,
                'char_count': len(sub),
            })

    doc.close()
    stats = {
        'n_pages' : n_pages,
        'n_chunks': len(chunks),
        'avg_chars': sum(c['char_count'] for c in chunks) // max(len(chunks), 1),
    }
    return chunks, stats


t0 = time.time()
chunks, stats = load_pdf_pymupdf_rawdict(PDF_PATH)
elapsed = (time.time() - t0) * 1000

print(f"pymupdf rawdict extraction : {elapsed:.0f}ms")
print(f"Pages                      : {stats['n_pages']}")
print(f"Chunks                     : {stats['n_chunks']}")
print(f"Avg chunk length           : {stats['avg_chars']} chars")


## Step 6 — Index

In [ ]:
print(f"Embedding {len(chunks)} chunks...")
t0   = time.time()
embs = embed_batch([c['text'] for c in chunks], label='[index]')

pts = [
    PointStruct(
        id=str(uuid.uuid4()), vector=embs[i],
        payload={
            'text'    : chunks[i]['text'],
            'metadata': {
                'page'      : chunks[i]['page'],
                'char_count': chunks[i]['char_count'],
                'source'    : 'climate.pdf',
                'pdf_lib'   : 'pymupdf-rawdict',
            }
        }
    )
    for i in range(len(chunks))
]
qdrant.upsert(collection_name=COLLECTION_NAME, points=pts)
print(f"Indexed {qdrant.get_collection(COLLECTION_NAME).points_count} vectors in {time.time()-t0:.1f}s")


## Step 7 — Two-Level Cache

**L1 (exact):** `hashlib.sha256(query)` → cached answer dict  
**L2 (semantic):** list of `(embedding, answer)` pairs; on query, compute cosine
similarity against all stored embeddings, return the best match if it exceeds
`SEMANTIC_THRESHOLD`.

Cosine similarity of two normalised vectors = their dot product.
Titan V2 embeddings are already L2-normalised (`normalize=True`), so
`sim = sum(a*b for a, b in zip(v1, v2))`.


In [ ]:
def cosine_sim(a: List[float], b: List[float]) -> float:
    return sum(x * y for x, y in zip(a, b))   # vectors are already L2-normalised


class RAGCache:
    def __init__(self, semantic_threshold: float = SEMANTIC_THRESHOLD):
        self.threshold  = semantic_threshold
        self._exact: Dict[str, Dict]               = {}   # hash -> entry
        self._semantic: List[Dict]                 = []   # list of {vec, entry}
        self.stats = {'exact_hits': 0, 'semantic_hits': 0, 'misses': 0,
                      'embed_calls': 0, 'llm_calls': 0}

    def _hash(self, query: str) -> str:
        return hashlib.sha256(query.strip().lower().encode()).hexdigest()

    def lookup_exact(self, query: str) -> Optional[Dict]:
        return self._exact.get(self._hash(query))

    def lookup_semantic(self, qvec: List[float]) -> Optional[Dict]:
        if not self._semantic:
            return None
        best_sim, best_entry = 0.0, None
        for item in self._semantic:
            s = cosine_sim(qvec, item['vec'])
            if s > best_sim:
                best_sim, best_entry = s, item['entry']
        if best_sim >= self.threshold:
            return {**best_entry, 'cache_sim': best_sim}
        return None

    def store(self, query: str, qvec: List[float], answer: str,
              latency_ms: float, n_chunks: int):
        entry = {'query': query, 'answer': answer,
                 'latency_ms': latency_ms, 'n_chunks': n_chunks}
        self._exact[self._hash(query)] = entry
        self._semantic.append({'vec': qvec, 'entry': entry})

    def size(self) -> int:
        return len(self._exact)


cache = RAGCache(semantic_threshold=SEMANTIC_THRESHOLD)
print(f"RAGCache ready  (semantic threshold={SEMANTIC_THRESHOLD})")


## Step 8 — Caching RAG Pipeline

In [ ]:
SYSTEM_PROMPT = """You are a knowledgeable assistant answering questions about a climate document.
Answer ONLY from the provided context. Be concise and precise.
If the answer is not in the context, say so explicitly.
"""


def caching_rag(question: str, verbose: bool = True) -> Dict:
    t0 = time.time()

    # L1 — exact cache
    hit = cache.lookup_exact(question)
    if hit:
        cache.stats['exact_hits'] += 1
        latency = (time.time() - t0) * 1000
        if verbose:
            print(f"[L1 EXACT HIT]  Q: {question}")
            print(f"                latency={latency:.0f}ms (saved embed+retrieve+llm)")
            print()
        return {**hit, 'cache_level': 'L1_exact', 'latency_ms': latency}

    # Embed query (needed for L2 + retrieval)
    cache.stats['embed_calls'] += 1
    qvec = embed_text(question)
    t_embed = (time.time() - t0) * 1000

    # L2 — semantic cache
    hit = cache.lookup_semantic(qvec)
    if hit:
        cache.stats['semantic_hits'] += 1
        latency = (time.time() - t0) * 1000
        if verbose:
            print(f"[L2 SEMANTIC HIT]  sim={hit['cache_sim']:.4f}  Q: {question}")
            print(f"                   latency={latency:.0f}ms (saved retrieve+llm)")
            print()
        return {**hit, 'cache_level': 'L2_semantic', 'latency_ms': latency}

    # Cache miss — full pipeline
    cache.stats['misses'] += 1
    cache.stats['llm_calls'] += 1

    resp = qdrant.query_points(
        collection_name=COLLECTION_NAME, query=qvec,
        limit=TOP_K, with_payload=True)
    hits    = [p.payload['text'] for p in resp.points]
    context = '\n\n'.join(f"[{i+1}]\n{h}" for i, h in enumerate(hits))
    answer  = call_llm(SYSTEM_PROMPT, f"Context:\n{context}\n\nQuestion: {question}")
    latency = (time.time() - t0) * 1000

    cache.store(question, qvec, answer, latency, len(hits))

    if verbose:
        print(f"[CACHE MISS]  Q: {question}")
        print(f"              latency={latency:.0f}ms  (embed={t_embed:.0f}ms)")
        print(f"              A: {answer[:300]}")
        print()

    return {'query': question, 'answer': answer, 'cache_level': 'miss',
            'latency_ms': latency, 'n_chunks': len(hits)}


print("caching_rag() defined.")


## Step 9 — Cache Demo

Round 1: 4 distinct questions → all misses (populate the cache)  
Round 2: exact repeats → L1 hits  
Round 3: paraphrased versions → L2 semantic hits  


In [ ]:
# Round 1 — populate cache
print("=" * 70)
print("ROUND 1 — Cold queries (expected: all misses)")
print("=" * 70)
cold_questions = [
    "What is Numerical Weather Prediction?",
    "What types of satellites are used for weather observation?",
    "What are the five weather forecasting methods?",
    "How is weather observation data collected globally?",
]
r1 = [caching_rag(q, verbose=True) for q in cold_questions]
print(f"Cache size after round 1: {cache.size()} entries")
print()

# Round 2 — exact repeats
print("=" * 70)
print("ROUND 2 — Exact repeats (expected: L1 hits)")
print("=" * 70)
r2 = [caching_rag(q, verbose=True) for q in cold_questions]

# Round 3 — paraphrased queries
print("=" * 70)
print("ROUND 3 — Paraphrased queries (expected: L2 semantic hits)")
print("=" * 70)
paraphrased = [
    "Explain Numerical Weather Prediction and how it works.",
    "Which satellites does weather forecasting rely on?",
    "List the forecasting techniques described in the document.",
    "How do meteorological stations gather data around the world?",
]
r3 = [caching_rag(q, verbose=True) for q in paraphrased]


## Step 10 — Cache Statistics

In [ ]:
total_queries = (cache.stats['exact_hits'] +
                 cache.stats['semantic_hits'] +
                 cache.stats['misses'])
hit_rate = (cache.stats['exact_hits'] + cache.stats['semantic_hits']) / total_queries * 100

print("Cache statistics")
print("-" * 35)
print(f"Total queries     : {total_queries}")
print(f"L1 exact hits     : {cache.stats['exact_hits']}")
print(f"L2 semantic hits  : {cache.stats['semantic_hits']}")
print(f"Cache misses      : {cache.stats['misses']}")
print(f"Overall hit rate  : {hit_rate:.0f}%")
print(f"Embed calls made  : {cache.stats['embed_calls']}  (saved {total_queries - cache.stats['embed_calls']})")
print(f"LLM calls made    : {cache.stats['llm_calls']}  (saved {total_queries - cache.stats['llm_calls']})")
print()

all_results = r1 + r2 + r3
miss_latency    = [r['latency_ms'] for r in all_results if r['cache_level'] == 'miss']
l1_latency      = [r['latency_ms'] for r in all_results if r['cache_level'] == 'L1_exact']
l2_latency      = [r['latency_ms'] for r in all_results if r['cache_level'] == 'L2_semantic']

def avg(lst): return sum(lst)/len(lst) if lst else 0

print(f"Avg latency — miss      : {avg(miss_latency):.0f}ms")
print(f"Avg latency — L2 hit    : {avg(l2_latency):.0f}ms")
print(f"Avg latency — L1 hit    : {avg(l1_latency):.2f}ms")
if miss_latency and l1_latency:
    print(f"L1 speedup vs miss      : {avg(miss_latency)/avg(l1_latency):.0f}×")
if miss_latency and l2_latency:
    print(f"L2 speedup vs miss      : {avg(miss_latency)/avg(l2_latency):.1f}×")


## Step 11 — Summary

### Two-level cache design

| Level | Key | Cost on hit | When to use |
|-------|-----|------------|-------------|
| L1 exact | SHA-256 of query | O(1) dict lookup | FAQ / repeated identical queries |
| L2 semantic | Cosine similarity | 1 embed call + O(n) dot products | Paraphrased / near-duplicate queries |
| Miss | N/A | Full pipeline | Novel queries |

### Threshold tuning

| Threshold | Effect |
|-----------|--------|
| 0.99+ | Almost only typo variants hit L2 |
| 0.75 (default) | Close paraphrases and reformulations hit |
| 0.60 | Topically similar but different questions may hit — risk of wrong answers |

### pymupdf `get_text("rawdict")` vs other modes

| Mode | Char access | Typical use |
|------|------------|-------------|
| `"text"` | No | Fast plain dump |
| `"dict"` | Via spans | Font/size analysis |
| `"blocks"` | No | Block-type filter |
| `"rawdict"` | Yes — `char['c']` | Glyph-level control, ligature handling |

> **Next: [29 — Evaluation RAG](29_Evaluation_RAG.ipynb)** —
> measure retrieval and generation quality with precision, recall, faithfulness, and NDCG.


In [ ]:
print(f"Collection '{COLLECTION_NAME}': {qdrant.get_collection(COLLECTION_NAME).points_count} vectors")
print(f"PDF framework  : pymupdf get_text('rawdict') — char-level glyph extraction")
print(f"RAG pattern    : Caching RAG — L1 exact (SHA-256) + L2 semantic (cosine sim)")
print(f"Cache entries  : {cache.size()}")
print(f"Hit rate       : {hit_rate:.0f}%  (L1={cache.stats['exact_hits']} L2={cache.stats['semantic_hits']} miss={cache.stats['misses']})")
print()
print("Notebook 28 — Caching RAG complete.")
